# 24-788 Mini-Project: HOMO-LUMO Gap Prediction on QM9

**Task:** Predict the HOMO-LUMO energy gap (Δε) for small organic molecules.
**Dataset:** QM9 — ~130 k molecules with DFT-computed quantum properties.
**Metric:** Mean Absolute Error (MAE) in meV.

| Model | Reference |
|---|---|
| Baseline — GCN | Kipf & Welling, ICLR 2017 |
| Variant 1 — Graph Attention Network (GAT) | Veličković et al., ICLR 2019 |
| Variant 2 — Sparse GCN (RigL) | Evci et al., ICML 2020 |

## 1 · Setup

In [ ]:
# Mount Google Drive so checkpoints persist across free-Colab session restarts.
# If Drive is unavailable the notebook falls back to local /content storage.
import os

USE_DRIVE = True   # set False to skip Drive mount

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/qm9_project'
    except Exception:
        USE_DRIVE = False

if not USE_DRIVE:
    SAVE_DIR = '/content/qm9_project'

os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_DIR = os.path.join(SAVE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Save directory:', SAVE_DIR)

In [ ]:
# Install dependencies (run once per Colab session).
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip('torch-geometric')
pip('rigl-torch')
print('All packages installed.')

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
import numpy as np
import matplotlib.pyplot as plt
import json, os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
torch.manual_seed(42)
np.random.seed(42)
from torch.utils.tensorboard import SummaryWriter

In [ ]:
# TensorBoard — load extension and define log directory.
# Logs persist in Google Drive across sessions.
%load_ext tensorboard

LOG_DIR = os.path.join(SAVE_DIR, 'runs')
os.makedirs(LOG_DIR, exist_ok=True)
print(f"TensorBoard log directory: {LOG_DIR}")

## 2 · Configuration

In [ ]:
# Target
TARGET_IDX  = 4          # HOMO-LUMO gap index in QM9.y (units: eV)
TARGET_NAME = 'HOMO-LUMO gap'
TARGET_UNIT = 'eV'

# Architecture
IN_CHANNELS  = 11
HIDDEN_DIM   = 128
NUM_LAYERS   = 4

# Training
BATCH_SIZE   = 64
LR           = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS       = 50
PATIENCE     = 12
LR_PATIENCE  = 6

# Splits
N_TRAIN = 110_000
N_VAL   =  10_000

# RigL
RIGL_SPARSITY = 0.5
RIGL_DELTA    = 100
RIGL_ALPHA    = 0.3

print('Configuration set.')

## 3 · Data

In [ ]:
dataset = QM9(root=os.path.join(SAVE_DIR, 'data'))
print('Dataset size :', len(dataset), 'molecules')
print('Node features:', dataset.num_node_features)

In [ ]:
generator = torch.Generator().manual_seed(42)
perm = torch.randperm(len(dataset), generator=generator)

train_idx = perm[:N_TRAIN]
val_idx   = perm[N_TRAIN : N_TRAIN + N_VAL]
test_idx  = perm[N_TRAIN + N_VAL :]

all_y   = dataset.data.y
train_y = all_y[train_idx, TARGET_IDX]
MEAN    = train_y.mean().item()
STD     = train_y.std().item()

print('Target:', TARGET_NAME)
print('mean={:.4f} {}, std={:.4f} {}'.format(MEAN, TARGET_UNIT, STD, TARGET_UNIT))
print('Train/Val/Test: {}/{}/{}'.format(len(train_idx), len(val_idx), len(test_idx)))

In [ ]:
train_dataset = dataset[train_idx]
val_dataset   = dataset[val_idx]
test_dataset  = dataset[test_idx]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print('Batches per epoch:', len(train_loader))

## 4 · Baseline: Graph Convolutional Network (GCN)

Kipf & Welling, *Semi-supervised Classification with GCNs*, ICLR 2017.

GCN propagates node features by averaging neighbour representations. Molecules
are natural graphs (atoms=nodes, bonds=edges). Global mean pooling aggregates
node embeddings into a graph-level vector for the regression head.

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS,
                 hidden=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=0.1):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden, add_self_loops=True))
        self.bns.append(nn.BatchNorm1d(hidden))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden, hidden, add_self_loops=True))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = x.float()
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x).relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)

_tmp = GCN()
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print('GCN parameters:', n_params)
del _tmp

## 5 · Training Utilities

In [ ]:
def train_epoch(model, loader, optimizer, device,
                mean=None, std=None, target=TARGET_IDX):
    mean = mean if mean is not None else MEAN
    std  = std  if std  is not None else STD
    model.train()
    total_loss = 0.0
    for data in loader:
        data   = data.to(device)
        y_norm = (data.y[:, target] - mean) / std
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index, data.batch)
        loss = F.mse_loss(out, y_norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device, mean=None, std=None, target=TARGET_IDX):
    mean = mean if mean is not None else MEAN
    std  = std  if std  is not None else STD
    model.eval()
    mae_sum = 0.0
    for data in loader:
        data = data.to(device)
        out  = model(data.x, data.edge_index, data.batch)
        pred = out * std + mean
        true = data.y[:, target]
        mae_sum += (pred - true).abs().sum().item()
    return mae_sum / len(loader.dataset)


def run_training(model, train_loader, val_loader, optimizer, scheduler,
                 epochs=EPOCHS, patience=PATIENCE,
                 save_path=None, label='Model', train_fn=None,
                 writer=None, log_sparsity=False):
    if train_fn is None:
        train_fn = train_epoch
    best_val = float('inf')
    wait     = 0
    history  = {'train_loss': [], 'val_mae_mev': []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, device=DEVICE)
        epoch_secs = time.time() - t0
        val_mae    = evaluate(model, val_loader, device=DEVICE)
        lr_now = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step(val_mae)
        history['train_loss'].append(train_loss)
        history['val_mae_mev'].append(val_mae * 1000)
        if writer is not None:
            writer.add_scalar('Loss/train',   train_loss,      epoch)
            writer.add_scalar('MAE/val_meV',  val_mae * 1000,  epoch)
            writer.add_scalar('LR',           lr_now,          epoch)
            writer.add_scalar('Time/epoch_sec', epoch_secs,    epoch)
            if log_sparsity:
                tot = sum(p.numel() for p in model.parameters() if p.dim() > 1)
                nz  = sum((p != 0).sum().item() for p in model.parameters() if p.dim() > 1)
                writer.add_scalar('Sparsity', 1.0 - nz / max(tot, 1), epoch)
        if val_mae < best_val:
            best_val = val_mae
            wait = 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= patience:
                print('  [{}] Early stopping at epoch {}.'.format(label, epoch))
                break
        if epoch % 5 == 0 or epoch == 1:
            print('[{}] Ep {:03d} | loss {:.5f} | val {:.2f} meV | lr {:.2e}'.format(
                  label, epoch, train_loss, val_mae * 1000, lr_now))
    return history, best_val

print('Utilities defined.')

## 6 · Train Baseline GCN

In [ ]:
model_baseline = GCN().to(DEVICE)
opt_b = torch.optim.Adam(model_baseline.parameters(),
                          lr=LR, weight_decay=WEIGHT_DECAY)
sch_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_b, patience=LR_PATIENCE, factor=0.5, min_lr=1e-5)

BASELINE_CKPT = os.path.join(CKPT_DIR, 'baseline_best.pt')
writer_b = SummaryWriter(os.path.join(LOG_DIR, 'baseline'))
print('=' * 65)
print('Baseline GCN')
print('=' * 65)

hist_baseline, _ = run_training(
    model_baseline, train_loader, val_loader, opt_b, sch_b,
    epochs=EPOCHS, patience=PATIENCE,
    save_path=BASELINE_CKPT, label='Baseline',
    writer=writer_b)

In [ ]:
model_baseline.load_state_dict(torch.load(BASELINE_CKPT))
test_mae_baseline = evaluate(model_baseline, test_loader, DEVICE)
print('Baseline GCN — Test MAE: {:.2f} meV'.format(test_mae_baseline * 1000))
writer_b.add_scalar('MAE/test_meV', test_mae_baseline * 1000, 0)
writer_b.close()

with open(os.path.join(SAVE_DIR, 'hist_baseline.json'), 'w') as f:
    json.dump(hist_baseline, f)
print('History saved.')

## 7 · Variant 1: Graph Attention Network (GAT)

Veličković et al., *"Graph Attention Networks"*, **ICLR 2019**.

**Hypothesis:** standard GCN averages all neighbour features with the same weight,
treating every bond equally. In real molecules, different bond types (single, double,
aromatic) contribute differently to electronic properties. GAT learns a separate
attention coefficient for each directed edge during training, so the model can
selectively up-weight or down-weight neighbours based on their feature compatibility.
For HOMO-LUMO prediction — which is sensitive to conjugation patterns and specific
functional groups — this learned weighting may improve accuracy with no additional
parameters beyond the multi-head projection.

`GATConv` is part of PyTorch Geometric, so no new dependencies are needed.

In [ ]:
from torch_geometric.nn import GATConv

class GAT(nn.Module):
    """
    Graph Attention Network for graph-level regression.

    Architecture:
        [GATConv(heads=4, concat=False) -> BatchNorm -> ReLU -> Dropout] x num_layers
        -> GlobalMeanPool
        -> Linear(hidden, hidden//2) -> ReLU -> Dropout -> Linear(hidden//2, 1)

    concat=False averages attention heads so channel width stays at `hidden`
    throughout, matching the baseline GCN parameter count closely.
    """
    def __init__(self,
                 in_channels=IN_CHANNELS,
                 hidden=HIDDEN_DIM,
                 num_layers=NUM_LAYERS,
                 heads=4,
                 dropout=0.1):
        super().__init__()
        self.dropout = dropout

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden, heads=heads,
                                  concat=False, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden))
        for _ in range(num_layers - 1):
            self.convs.append(GATConv(hidden, hidden, heads=heads,
                                      concat=False, dropout=dropout))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = x.float()
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x).relu()
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)

_tmp = GAT()
n_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print('GAT parameters: {:,}'.format(n_params))
del _tmp

# ── Train ─────────────────────────────────────────────────────────────────────
model_gat = GAT().to(DEVICE)
opt_g = torch.optim.Adam(model_gat.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sch_g = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_g, patience=LR_PATIENCE, factor=0.5, min_lr=1e-5)

GAT_CKPT = os.path.join(CKPT_DIR, 'gat_best.pt')
writer_g = SummaryWriter(os.path.join(LOG_DIR, 'gat'))

print('=' * 65)
print('Variant 1 - Graph Attention Network (GAT)')
print('=' * 65)

hist_gat, best_val_gat = run_training(
    model_gat, train_loader, val_loader,
    opt_g, sch_g,
    epochs=EPOCHS, patience=PATIENCE,
    save_path=GAT_CKPT,
    label='GAT',
    writer=writer_g,
)

In [ ]:
model_gat.load_state_dict(torch.load(GAT_CKPT))
test_mae_gat = evaluate(model_gat, test_loader, DEVICE)
print('\nGAT  —  Test MAE: {:.2f} meV'.format(test_mae_gat * 1000))

writer_g.add_scalar('MAE/test_meV', test_mae_gat * 1000, 0)
writer_g.close()

with open(os.path.join(SAVE_DIR, 'hist_gat.json'), 'w') as f:
    json.dump(hist_gat, f)
print('History saved -> hist_gat.json')

## 8 · Variant 2: Sparse GCN (RigL)

Evci et al., *Rigging the Lottery: Making All Tickets Winners*, ICML 2020.

**Hypothesis:** many GCN weights on QM9 are redundant — molecules are small
(up to 29 heavy atoms) and structural motifs are sparse. RigL trains a sparse
network end-to-end: it removes low-magnitude weights and regrows connections
where gradients are largest, letting the topology evolve during training.
A 50% sparse network also halves memory and FLOPs at inference.

In [ ]:
from rigl_torch.RigL import RigLScheduler

model_sparse = GCN().to(DEVICE)
opt_s        = torch.optim.Adam(model_sparse.parameters(),
                                lr=LR, weight_decay=WEIGHT_DECAY)

T_end = int(0.75 * len(train_loader) * EPOCHS)

pruner = RigLScheduler(
    model_sparse, opt_s,
    dense_allocation      = 1.0 - RIGL_SPARSITY,
    sparsity_distribution = 'uniform',
    T_end                 = T_end,
    delta                 = RIGL_DELTA,
    alpha                 = RIGL_ALPHA,
    static_topo           = False,
    grad_accumulation_n   = 1,
    ignore_linear_layers  = False,
    keep_first_layer_dense= True,
)

sch_s = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_s, patience=LR_PATIENCE, factor=0.5, min_lr=1e-5)

def train_epoch_rigl(model, loader, optimizer, device,
                     mean=None, std=None, target=TARGET_IDX):
    mean = mean if mean is not None else MEAN
    std  = std  if std  is not None else STD
    model.train()
    total_loss = 0.0
    for data in loader:
        data   = data.to(device)
        y_norm = (data.y[:, target] - mean) / std
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index, data.batch)
        loss = F.mse_loss(out, y_norm)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        if pruner():
            optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

SPARSE_CKPT = os.path.join(CKPT_DIR, 'sparse_best.pt')
writer_s = SummaryWriter(os.path.join(LOG_DIR, 'sparse'))
print('=' * 65)
print('Variant 2 - Sparse GCN (RigL, {}% sparsity)'.format(
      int(RIGL_SPARSITY * 100)))
print('=' * 65)

hist_sparse, _ = run_training(
    model_sparse, train_loader, val_loader, opt_s, sch_s,
    epochs=EPOCHS, patience=PATIENCE,
    save_path=SPARSE_CKPT, label='Sparse',
    train_fn=train_epoch_rigl,
    writer=writer_s, log_sparsity=True)

In [ ]:
model_sparse.load_state_dict(torch.load(SPARSE_CKPT))
test_mae_sparse = evaluate(model_sparse, test_loader, DEVICE)
writer_s.add_scalar('MAE/test_meV', test_mae_sparse * 1000, 0)
writer_s.close()
print('Sparse GCN — Test MAE: {:.2f} meV'.format(test_mae_sparse * 1000))

with open(os.path.join(SAVE_DIR, 'hist_sparse.json'), 'w') as f:
    json.dump(hist_sparse, f)
print('History saved.')

## 8b · TensorBoard — Inspect Training Runs

Run the cell below after training starts to open the interactive dashboard.
All three runs are logged to `LOG_DIR/runs/` and persist in Google Drive.

In [ ]:
%tensorboard --logdir {LOG_DIR}

## 9 · Results and Analysis

In [ ]:
SOTA_MEV = 50.0

results = {
    'Baseline GCN'      : test_mae_baseline   * 1000,
    'GAT'     : test_mae_gat  * 1000,
    'Sparse GCN (RigL)' : test_mae_sparse     * 1000,
}

print('=' * 55)
for name, mae in results.items():
    print('{:<24} {:.2f} meV'.format(name, mae))
print('-' * 55)
print('{:<24} {:.1f} meV  (SotA ref.)'.format('', SOTA_MEV))
print('=' * 55)

with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
    json.dump({k: round(v, 3) for k, v in results.items()}, f, indent=2)
print('Results saved.')

In [ ]:
# Learning-curve and bar-chart figure (used in the report).
COLORS = {
    'Baseline GCN'      : '#4C72B0',
    'GAT'     : '#DD8452',
    'Sparse GCN (RigL)' : '#55A868',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for lbl, hist in [('Baseline GCN',      hist_baseline),
                  ('GAT',      hist_gat),
                  ('Sparse GCN (RigL)',  hist_sparse)]:
    ax.plot(hist['val_mae_mev'], label=lbl, color=COLORS[lbl], linewidth=2)
ax.axhline(SOTA_MEV, color='gray', linestyle='--',
           linewidth=1.2, label='SotA ref.')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MAE (meV)')
ax.set_title('Validation MAE vs. Epoch')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
names  = list(results.keys())
maes   = list(results.values())
colors = [COLORS[n] for n in names]
bars   = ax.bar(names, maes, color=colors,
                alpha=0.85, edgecolor='black', linewidth=0.8)
ax.axhline(SOTA_MEV, color='gray', linestyle='--',
           linewidth=1.2, label='SotA ref.')
for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width() / 2.,
            bar.get_height() + 0.4,
            '{:.1f}'.format(mae),
            ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Test MAE (meV)')
ax.set_title('Test MAE Comparison')
ax.set_ylim(0, max(maes) * 1.35)
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=10, ha='right')

plt.tight_layout()
FIG_PATH = os.path.join(SAVE_DIR, 'results_comparison.pdf')
plt.savefig(FIG_PATH, dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved:', FIG_PATH)

### Sparsity Analysis (RigL)

Layer-wise weight density after training — useful for the Results section.

In [ ]:
print('{:<40} {:>10} {:>10} {:>9}'.format(
      'Layer', 'Non-zero', 'Total', 'Density'))
print('-' * 73)
total_nz, total_params = 0, 0
for name, param in model_sparse.named_parameters():
    if param.dim() > 1:
        nz      = (param != 0).sum().item()
        total   = param.numel()
        density = nz / total
        total_nz     += nz
        total_params += total
        print('{:<40} {:>10,} {:>10,} {:>8.1%}'.format(
              name, nz, total, density))
print('-' * 73)
overall = total_nz / total_params if total_params > 0 else 0
print('{:<40} {:>10,} {:>10,} {:>8.1%}'.format(
      'Overall', total_nz, total_params, overall))

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Parameter counts:')
print('  Baseline GCN  :', count_params(model_baseline))
print('  GAT :', count_params(model_gat),
      '(includes gat branches)')
print('  Sparse GCN    :', count_params(model_sparse),
      '(~{}% zeroed)'.format(int(RIGL_SPARSITY * 100)))

## 10 · Reproduce Results  *(standalone — run without retraining)*

**This section satisfies the `reproduce_results` submission requirement (§5.3).**

Fully self-contained given only the saved checkpoints and history JSON files.
It regenerates all reported **metrics** and all **figures** without any training.

After a session restart:
1. Run cells 1–5 (setup, installs, imports, config, data).
2. Jump straight here and run the two cells below.

In [ ]:
# 10a. Load checkpoints and re-evaluate on test set
# Requires: checkpoints saved to CKPT_DIR during training.
# Run cells 1-5 first (setup, installs, imports, config, data), then this cell.

print("=" * 65)
print("Reproducing results from saved checkpoints")
print("=" * 65)

r_baseline  = GCN().to(DEVICE)
r_gat = GCN().to(DEVICE)
r_sparse    = GCN().to(DEVICE)

r_baseline.load_state_dict(
    torch.load(os.path.join(CKPT_DIR, "baseline_best.pt"),
               map_location=DEVICE))
r_gat.load_state_dict(
    torch.load(os.path.join(CKPT_DIR, "gat_best.pt"),
               map_location=DEVICE))
r_sparse.load_state_dict(
    torch.load(os.path.join(CKPT_DIR, "sparse_best.pt"),
               map_location=DEVICE))

repro_results = {
    "Baseline GCN"      : evaluate(r_baseline,  test_loader, DEVICE) * 1000,
    "GAT"     : evaluate(r_gat, test_loader, DEVICE) * 1000,
    "Sparse GCN (RigL)" : evaluate(r_sparse,    test_loader, DEVICE) * 1000,
}

SOTA_MEV = 50.0
header = "{:<24} {:>14} {:>13}".format(
    "Model", "Test MAE (meV)", "vs Baseline")
print("\n" + header)
print("-" * 55)
baseline_mae = repro_results["Baseline GCN"]
for name, mae in repro_results.items():
    if name == "Baseline GCN":
        delta_str = "    -"
    else:
        delta_str = "{:+.2f} meV".format(mae - baseline_mae)
    print("{:<24} {:>14.2f} {:>13}".format(name, mae, delta_str))
print("-" * 55)
print("{:<24} {:>14.1f}".format("SotA (GNN reference)", SOTA_MEV))


In [ ]:
# 10b. Regenerate all figures from saved history files
# Requires: hist_baseline.json, hist_gat.json, hist_sparse.json
# (written automatically after each model's training cell).

with open(os.path.join(SAVE_DIR, "hist_baseline.json"))  as f:
    rh_b = json.load(f)
with open(os.path.join(SAVE_DIR, "hist_gat.json")) as f:
    rh_d = json.load(f)
with open(os.path.join(SAVE_DIR, "hist_sparse.json"))    as f:
    rh_s = json.load(f)

COLORS = {
    "Baseline GCN"      : "#4C72B0",
    "GAT"     : "#DD8452",
    "Sparse GCN (RigL)" : "#55A868",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Validation MAE learning curves
ax = axes[0]
pairs = [
    ("Baseline GCN",      rh_b),
    ("GAT",     rh_d),
    ("Sparse GCN (RigL)", rh_s),
]
for label, hist in pairs:
    ax.plot(hist["val_mae_mev"], label=label,
            color=COLORS[label], linewidth=2)
ax.axhline(SOTA_MEV, color="gray", linestyle="--",
           linewidth=1.2, label="SotA ref. (50 meV)")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Validation MAE (meV)", fontsize=12)
ax.set_title("Validation MAE vs. Epoch", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: Test MAE bar chart
ax = axes[1]
names  = list(repro_results.keys())
maes   = list(repro_results.values())
colors = [COLORS[n] for n in names]
bars   = ax.bar(names, maes, color=colors,
                alpha=0.85, edgecolor="black", linewidth=0.8)
ax.axhline(SOTA_MEV, color="gray", linestyle="--",
           linewidth=1.2, label="SotA ref.")
for bar, mae in zip(bars, maes):
    ax.text(
        bar.get_x() + bar.get_width() / 2.,
        bar.get_height() + 0.4,
        "{:.1f}".format(mae),
        ha="center", va="bottom", fontweight="bold", fontsize=11,
    )
ax.set_ylabel("Test MAE (meV)", fontsize=12)
ax.set_title("Test MAE Comparison", fontsize=13)
ax.set_ylim(0, max(maes) * 1.35)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=10, ha="right", fontsize=10)

plt.tight_layout()
FIG_PATH = os.path.join(SAVE_DIR, "results_comparison.pdf")
plt.savefig(FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: " + FIG_PATH)
